In [1]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
events = pd.read_csv("events.csv")
events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [5]:
events.shape

(2756101, 5)

In [6]:
events.columns

Index(['timestamp', 'visitorid', 'event', 'itemid', 'transactionid'], dtype='object')

In [7]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          object 
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 105.1+ MB


In [8]:
events["event"].value_counts()

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [9]:
#check unique visitors
events["visitorid"].nunique()

1407580

In [10]:
events.groupby('event')['visitorid'].nunique()

event
addtocart        37722
transaction      11719
view           1404179
Name: visitorid, dtype: int64

In [11]:
events.isnull().sum()

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

In [12]:
events['datetime'] =  pd.to_datetime(events['timestamp'],unit="ms")

In [13]:
events[["timestamp", "datetime"]].head()

,timestamp,datetime
0,1433221332117,2015-06-02 05:02:12.117
1,1433224214164,2015-06-02 05:50:14.164
2,1433221999827,2015-06-02 05:13:19.827
3,1433221955914,2015-06-02 05:12:35.914
4,1433221337106,2015-06-02 05:02:17.106


In [15]:
#data range
print("Start date:", events['datetime'].min())
print("End date:", events["datetime"].max())

Start date: 2015-05-03 03:00:04.384000
End date: 2015-09-18 02:59:47.788000


In [22]:
funnel = events.groupby("event")["visitorid"].nunique()
funnel

event
addtocart        37722
transaction      11719
view           1404179
Name: visitorid, dtype: int64

In [20]:
#unique visitor counts
viewers = funnel["view"]
cart_users = funnel["addtocart"]
purchasers = funnel["transaction"]

view_to_cart =  cart_users/viewers * 100
cart_to_purchase = purchasers/cart_users * 100
view_to_purchase = purchasers/viewers * 100

print(f"view to Add to cart: {view_to_cart:.2f}%")
print(f"Add to cart to Purchase: {cart_to_purchase:.2f}%")
print(f"View to Purchase:{view_to_purchase:.2f}%")



view to Add to cart: 2.69%
Add to cart to Purchase: 31.07%
View to Purchase:0.83%


The largest bottleneck is View to Add to Cart
Approximately 97.3% of unique viewers did not add an item to their cart after viewing the product. 

In [23]:
funnel_data = pd.DataFrame({
    "Stage": ["Product View","Add to Cart", "Purchase"], 
    "Unique Visitors": [viewers, cart_users, purchasers]})
funnel_data["Conversion Rate"] = (funnel_data["Unique Visitors"]/viewers * 100)
funnel_data

,Stage,Unique Visitors,Conversion Rate
0,Product View,1404179,100.00000
1,Add to Cart,37722,2.68641
2,Purchase,11719,0.83458
